In [1]:
from IPython.display import display, Markdown, HTML

# Prepare translation pipeline results for SKM import

## Description
 - PSS -- create lists per functional cluster, to import to neo4j
 - CKN -- prepare table for CKN v2



# Setup

## Library import
We import all the required Python libraries

In [2]:
# IO
from pathlib import Path
import json

# Data manipulation
import pandas as pd
import numpy as np

# Visualizations
import matplotlib as plt
import seaborn as sns

# handy / other
from collections import defaultdict
import re, os, time
from datetime import datetime

In [3]:
today = datetime.today().strftime('%Y-%m-%d'); today

'2026-04-24'

## Path definitions

In [4]:
base_dir = Path("../../")
input_dir = base_dir / "input" # translation pipeline input
output_dir = base_dir / "output" # translation pipeline output
skm_downloads_dir = base_dir / "skm_uploads" / "download_page"
skm_pss_translations_dir = base_dir / "skm_uploads" / "pss_db"

In [5]:
skm_downloads_dir.mkdir(exist_ok=True)
skm_pss_translations_dir.mkdir(exist_ok=True)

# Data processing

In [6]:
TRANSLATION_VERSION = "v20250924"
BINARY_EVIDENCE_COLS = ['FastOMA', 'MCScanX', 'OrthoDB', 'PLAZA', 'RBH', 'compara']

In [7]:
species_table = pd.read_csv(input_dir / "species_table.tsv", sep="\t"); species_table

,species,common_name,code,reference
0,Arabidopsis thaliana,thale cress,ath,ARAPORT11
1,Malus domestica,apple,mdo,Golden Delicious GDDH13 v1.1
2,"Prunus amygdalus, Prunus dulcis",almond,pdul,"Texas v3.0, F1"
3,Prunus armeniaca,apricot,parm,Marouch n14 v1.0
4,Prunus avium,wild cherry,pavi,Tieton v2.0
5,Prunus cerasifera,cherry plum,pcer,Montmorency v1.0.a2
6,Prunus persica,peach,ppe,Lovell v2.0.a1
7,Prunus sibirica,siberian apricot,psib,F106 v1.0
8,Pyrus communis,pear,pcox,"d'Anjou v2.3.a1, hap 1"
9,Solanum lycopersicum,tomato,sly,Heinz 1706 ITAG 4.0


## Files for downloads page

In [8]:
def concat_evidence(df):
    ''' inplace function '''

    cols = [c for c in BINARY_EVIDENCE_COLS if c in df.columns]
    
    df["evidence"] = (
        df[cols]
        .astype(bool)
        .dot(pd.Index(cols) + ",")
        .str.rstrip(",")
        .str.split(",")
    )
    
    df["evidence"] = df.apply(
        lambda row: row["evidence"] + (["MapMan4_Match"] if row["MapMan4_Match"] != "no match" else []),
        axis=1
    )
    
    df["evidence"] = df["evidence"].apply(lambda x: ",".join(x))

In [9]:
for f in output_dir.glob("*orthologues-kept_2025-09-24-strict.xlsx"):
    print(f)
    species_to, species_from = f.name.split("_")[0].split("-")

../../output/apricot-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/wildcherry-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/peach-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/potato-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/almond-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/grapevine-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/pear-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/siberianapricot-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/cherryplum-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/apple-ath_orthologues-kept_2025-09-24-strict.xlsx
../../output/tomato-ath_orthologues-kept_2025-09-24-strict.xlsx


In [10]:
for f in output_dir.glob("*orthologues-kept_2025-09-24-strict.xlsx"):
    species_to, species_from = f.name.split("_")[0].split("-")
    print(species_from, species_to, f)
        
    df = pd.read_excel(f)
    concat_evidence(df)

    df = df[['from_geneID', 'to_geneID', 'athName', 'filter_criteria', 'evidence']]

    df.rename(columns={
        "from_geneID": f"{species_from}_source", 
        "to_geneID": species_to,
        "athName":"symbol_source"
    }, inplace=True)

    f_out = skm_downloads_dir / f"translation_{species_from}_to_{species_to}_{TRANSLATION_VERSION}.tsv.gz"
    df.to_csv(f_out, sep="\t", index=False, compression='gzip')

    species_table.loc[(species_table['common_name'].str.replace(" ", "") == species_to), "file_name"] = str(f_out.name)

ath apricot ../../output/apricot-ath_orthologues-kept_2025-09-24-strict.xlsx
ath wildcherry ../../output/wildcherry-ath_orthologues-kept_2025-09-24-strict.xlsx
ath peach ../../output/peach-ath_orthologues-kept_2025-09-24-strict.xlsx
ath potato ../../output/potato-ath_orthologues-kept_2025-09-24-strict.xlsx
ath almond ../../output/almond-ath_orthologues-kept_2025-09-24-strict.xlsx
ath grapevine ../../output/grapevine-ath_orthologues-kept_2025-09-24-strict.xlsx
ath pear ../../output/pear-ath_orthologues-kept_2025-09-24-strict.xlsx
ath siberianapricot ../../output/siberianapricot-ath_orthologues-kept_2025-09-24-strict.xlsx
ath cherryplum ../../output/cherryplum-ath_orthologues-kept_2025-09-24-strict.xlsx
ath apple ../../output/apple-ath_orthologues-kept_2025-09-24-strict.xlsx
ath tomato ../../output/tomato-ath_orthologues-kept_2025-09-24-strict.xlsx


In [11]:
d = species_table.set_index("code").to_dict(orient="index"); d  # use to generate file endpoint code

{'ath': {'species': 'Arabidopsis thaliana',
  'common_name': 'thale cress',
  'reference': 'ARAPORT11',
  'file_name': nan},
 'mdo': {'species': 'Malus domestica',
  'common_name': 'apple',
  'reference': 'Golden Delicious GDDH13 v1.1',
  'file_name': 'translation_ath_to_apple_v20250924.tsv.gz'},
 'pdul': {'species': 'Prunus amygdalus, Prunus dulcis',
  'common_name': 'almond',
  'reference': 'Texas v3.0, F1',
  'file_name': 'translation_ath_to_almond_v20250924.tsv.gz'},
 'parm': {'species': 'Prunus armeniaca',
  'common_name': 'apricot',
  'reference': 'Marouch n14 v1.0',
  'file_name': 'translation_ath_to_apricot_v20250924.tsv.gz'},
 'pavi': {'species': 'Prunus avium',
  'common_name': 'wild cherry',
  'reference': 'Tieton v2.0',
  'file_name': 'translation_ath_to_wildcherry_v20250924.tsv.gz'},
 'pcer': {'species': 'Prunus cerasifera',
  'common_name': 'cherry plum',
  'reference': 'Montmorency v1.0.a2',
  'file_name': 'translation_ath_to_cherryplum_v20250924.tsv.gz'},
 'ppe': {'spec

## Files for PSS db


ath_id \t vvi_homologues \t ... 

In [12]:
name2code = {}
for name, code in species_table.set_index("common_name")['code'].to_dict().items():
    name2code[name.replace(" ", "")] = code
name2code

{'thalecress': 'ath',
 'apple': 'mdo',
 'almond': 'pdul',
 'apricot': 'parm',
 'wildcherry': 'pavi',
 'cherryplum': 'pcer',
 'peach': 'ppe',
 'siberianapricot': 'psib',
 'pear': 'pcox',
 'tomato': 'sly',
 'potato': 'stu',
 'grapevine': 'vvi'}

In [13]:
dfs = []
for f in output_dir.glob("*orthologues-kept_2025-09-24-strict.xlsx"):
    species_to, species_from = f.name.split("_")[0].split("-")
    print(species_from, species_to, f)
        
    df = pd.read_excel(f, usecols=["from_geneID", "to_geneID"])

    df.rename(columns={
        "from_geneID": f"{species_from}_source", 
        "to_geneID": species_to,
    }, inplace=True)

    df = df.groupby("ath_source").agg(lambda x: ",".join(list(x)))
    df.columns = [f"{name2code[species_to.replace(" ", "")]}_homologues"]

    df.sort_index(inplace=True)
    
    dfs.append(df)

ath apricot ../../output/apricot-ath_orthologues-kept_2025-09-24-strict.xlsx
ath wildcherry ../../output/wildcherry-ath_orthologues-kept_2025-09-24-strict.xlsx
ath peach ../../output/peach-ath_orthologues-kept_2025-09-24-strict.xlsx
ath potato ../../output/potato-ath_orthologues-kept_2025-09-24-strict.xlsx
ath almond ../../output/almond-ath_orthologues-kept_2025-09-24-strict.xlsx
ath grapevine ../../output/grapevine-ath_orthologues-kept_2025-09-24-strict.xlsx
ath pear ../../output/pear-ath_orthologues-kept_2025-09-24-strict.xlsx
ath siberianapricot ../../output/siberianapricot-ath_orthologues-kept_2025-09-24-strict.xlsx
ath cherryplum ../../output/cherryplum-ath_orthologues-kept_2025-09-24-strict.xlsx
ath apple ../../output/apple-ath_orthologues-kept_2025-09-24-strict.xlsx
ath tomato ../../output/tomato-ath_orthologues-kept_2025-09-24-strict.xlsx


In [72]:
df_all = pd.concat(dfs, axis=1)
df_all

,parm_homologues,pavi_homologues,ppe_homologues,stu_homologues,pdul_homologues,vvi_homologues,pcox_homologues,psib_homologues,pcer_homologues,mdo_homologues,sly_homologues
ath_source,,,,,,,,,,,
AT1G01020,PruarM.2G365300,FUN_011749,Prupe.2G201100,Soltu.DM.11G003840,TexasF1_G9059,Vitvi05_01chr15g15160,"Pyrco.da.v2a1.chr1A.345990,Pyrco.da.v2a1.chr7A...",PaF106G0200009361,"Pcer_051687-RA,Pcer_070038-RA,Pcer_074823-RA,P...","Maldo.hc.v1a1.ch1A.g25188,Maldo.hc.v1a1.ch7A.g...",NaN
AT1G01030,"PruarM.2G365200,PruarM.5G193300",FUN_011749,"Prupe.2G201000,Prupe.5G134900","Soltu.DM.08G005020,Soltu.DM.08G005030,Soltu.DM...",TexasF1_G18833,"Vitvi05_01chr02g03430,Vitvi05_01chr15g15190","Pyrco.da.v2a1.chr14A.371380,Pyrco.da.v2a1.chr1...","PaF106G0200009360,PaF106G0500020091","Pcer_051686-RA,Pcer_064305-RA,Pcer_070037-RA,P...","Maldo.hc.v1a1.ch14A.g13279,Maldo.hc.v1a1.ch1A....",Solyc08T002588
AT1G01040,PruarM.2G365100,FUN_011748,Prupe.2G200900,Soltu.DM.10G000330,TexasF1_G9057,Vitvi05_01chr15g15200,"Pyrco.da.v2a1.chr1A.345950,Pyrco.da.v2a1.chr7A...","PaF106G0200009357,PaF106G0200009358","Pcer_051685-RA,Pcer_070036-RA,Pcer_074821-RA,P...",Maldo.hc.v1a1.ch1A.g25186,Solyc10T000015
AT1G01050,"PruarM.2G364900,PruarM.2G365000","FUN_011739,FUN_011747","Prupe.2G200700,Prupe.2G200800","Soltu.DM.08G005040,Soltu.DM.08G030280,Soltu.DM...","TexasF1_G9055,TexasF1_G9056","Vitvi05_01chr02g03390,Vitvi05_01chr15g15220","Pyrco.da.v2a1.chr1A.345930,Pyrco.da.v2a1.chr1A...","PaF106G0200009355,PaF106G0200009356","Pcer_051683-RA,Pcer_051684-RA,Pcer_070034-RA,P...","Maldo.hc.v1a1.ch1A.g25184,Maldo.hc.v1a1.ch1A.g...","Solyc08T002584,Solyc12T002169"
AT1G01060,"PruarM.2G364300,PruarM.2G364500,PruarM.2G364600","FUN_011734,FUN_011735","Prupe.2G200300,Prupe.2G200400","Soltu.DM.02G004510,Soltu.DM.03G019030","TexasF1_G9050,TexasF1_G9051,TexasF1_G9052",Vitvi05_01chr15g15250,"Pyrco.da.v2a1.chr1A.345900,Pyrco.da.v2a1.snap....","PaF106G0200009351,PaF106G0200009352","Pcer_051679-RA,Pcer_051680-RA,Pcer_070026-RA,P...","Maldo.hc.v1a1.ch1A.g25174,Maldo.hc.v1a1.ch1A.g...",NaN
...,...,...,...,...,...,...,...,...,...,...,...
AT5G53820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Solyc02T001952
AT5G66220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Solyc05T000550
AT5G66990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Solyc02T000133,Solyc05T001201"


In [73]:
# df_all.rename(columns={
#     "stu_homologues": f"stu-unitato_homologues", 
# }, inplace=True)

In [74]:
df_all.notna().sum(axis=0)

parm_homologues    16773
pavi_homologues    17405
ppe_homologues     19239
stu_homologues     18684
pdul_homologues    18482
vvi_homologues     19002
pcox_homologues    17474
psib_homologues    15980
pcer_homologues    17856
mdo_homologues     19719
sly_homologues     18491
dtype: int64

In [75]:
df_all.to_csv(skm_pss_translations_dir / f"ath_translations_{TRANSLATION_VERSION}.tsv", sep="\t")

In [77]:
def split_str_columns(df: pd.DataFrame, cols=None) -> pd.DataFrame:
    if cols is None:
        cols = df.columns

    def split_cell(x):
        if pd.isna(x):
            return []
        x = x.upper()
        if isinstance(x, (list, tuple, np.ndarray)):
            return list(x)

        return [i for i in str(x).split(",") if i != ""]

    for c in cols:
        df[c] = df[c].apply(split_cell)

    return df
df_all = split_str_columns(df_all)

In [78]:
df_all.head()

,parm_homologues,pavi_homologues,ppe_homologues,stu_homologues,pdul_homologues,vvi_homologues,pcox_homologues,psib_homologues,pcer_homologues,mdo_homologues,sly_homologues
ath_source,,,,,,,,,,,
AT1G01020,[PRUARM.2G365300],[FUN_011749],[PRUPE.2G201100],[SOLTU.DM.11G003840],[TEXASF1_G9059],[VITVI05_01CHR15G15160],"[PYRCO.DA.V2A1.CHR1A.345990, PYRCO.DA.V2A1.CHR...",[PAF106G0200009361],"[PCER_051687-RA, PCER_070038-RA, PCER_074823-R...","[MALDO.HC.V1A1.CH1A.G25188, MALDO.HC.V1A1.CH7A...",[]
AT1G01030,"[PRUARM.2G365200, PRUARM.5G193300]",[FUN_011749],"[PRUPE.2G201000, PRUPE.5G134900]","[SOLTU.DM.08G005020, SOLTU.DM.08G005030, SOLTU...",[TEXASF1_G18833],"[VITVI05_01CHR02G03430, VITVI05_01CHR15G15190]","[PYRCO.DA.V2A1.CHR14A.371380, PYRCO.DA.V2A1.CH...","[PAF106G0200009360, PAF106G0500020091]","[PCER_051686-RA, PCER_064305-RA, PCER_070037-R...","[MALDO.HC.V1A1.CH14A.G13279, MALDO.HC.V1A1.CH1...",[SOLYC08T002588]
AT1G01040,[PRUARM.2G365100],[FUN_011748],[PRUPE.2G200900],[SOLTU.DM.10G000330],[TEXASF1_G9057],[VITVI05_01CHR15G15200],"[PYRCO.DA.V2A1.CHR1A.345950, PYRCO.DA.V2A1.CHR...","[PAF106G0200009357, PAF106G0200009358]","[PCER_051685-RA, PCER_070036-RA, PCER_074821-R...",[MALDO.HC.V1A1.CH1A.G25186],[SOLYC10T000015]
AT1G01050,"[PRUARM.2G364900, PRUARM.2G365000]","[FUN_011739, FUN_011747]","[PRUPE.2G200700, PRUPE.2G200800]","[SOLTU.DM.08G005040, SOLTU.DM.08G030280, SOLTU...","[TEXASF1_G9055, TEXASF1_G9056]","[VITVI05_01CHR02G03390, VITVI05_01CHR15G15220]","[PYRCO.DA.V2A1.CHR1A.345930, PYRCO.DA.V2A1.CHR...","[PAF106G0200009355, PAF106G0200009356]","[PCER_051683-RA, PCER_051684-RA, PCER_070034-R...","[MALDO.HC.V1A1.CH1A.G25184, MALDO.HC.V1A1.CH1A...","[SOLYC08T002584, SOLYC12T002169]"
AT1G01060,"[PRUARM.2G364300, PRUARM.2G364500, PRUARM.2G36...","[FUN_011734, FUN_011735]","[PRUPE.2G200300, PRUPE.2G200400]","[SOLTU.DM.02G004510, SOLTU.DM.03G019030]","[TEXASF1_G9050, TEXASF1_G9051, TEXASF1_G9052]",[VITVI05_01CHR15G15250],"[PYRCO.DA.V2A1.CHR1A.345900, PYRCO.DA.V2A1.SNA...","[PAF106G0200009351, PAF106G0200009352]","[PCER_051679-RA, PCER_051680-RA, PCER_070026-R...","[MALDO.HC.V1A1.CH1A.G25174, MALDO.HC.V1A1.CH1A...",[]


In [79]:
# to the functional clusters...

# cannot download as not logged in... 
jsonl_path = skm_pss_translations_dir / f"pss-json-restricted-{today}.json"

def extract_nodes_with_cluster_id(jsonl_path):
    nodes = []

    with open(jsonl_path, "r") as f:
        for line in f:
            obj = json.loads(line)

            # APOC export formats vary; handle common patterns
            if obj.get("type") == "node" or "labels" in obj:
                props = obj.get("properties", obj)

                if "functional_cluster_id" in props and not "PlantAbstract" in obj.get("labels"):
                    d = {"functional_cluster_id": props["functional_cluster_id"]}
                    for p in props:
                        if p.endswith("_homologues") and not p.startswith("_all") and not p.startswith("all"):
                            d[p] = [s.upper() for s in props[p]]
                            
                    nodes.append(d)

    return nodes

pss_nodes = extract_nodes_with_cluster_id(jsonl_path)

In [80]:
pss_fcs = pd.DataFrame(pss_nodes).sort_values("functional_cluster_id").reset_index(drop=True)
pss_fcs.head()

,functional_cluster_id,ath_homologues,stu_homologues,sly_homologues,osa_homologues,nta_homologues,vvi_homologues
0,fc00005,[AT2G39350],NaN,NaN,NaN,NaN,NaN
1,fc00006,"[AT1G04580, AT2G27150, AT5G20960]",NaN,NaN,NaN,NaN,NaN
2,fc00007,"[AT2G30720, AT5G48370]",NaN,NaN,NaN,NaN,NaN
3,fc00008,"[AT1G77330, AT1G05010, AT1G62380, AT1G12010, A...",[SOTUB07G018820.1.1],NaN,NaN,NaN,NaN
4,fc00009,"[AT1G01480, AT2G22810, AT3G49700, AT3G61510, A...",NaN,NaN,NaN,NaN,NaN


In [81]:
pss_fcs.explode("ath_homologues").set_index("ath_homologues").head()

,functional_cluster_id,stu_homologues,sly_homologues,osa_homologues,nta_homologues,vvi_homologues
ath_homologues,,,,,,
AT2G39350,fc00005,NaN,NaN,NaN,NaN,NaN
AT1G04580,fc00006,NaN,NaN,NaN,NaN,NaN
AT2G27150,fc00006,NaN,NaN,NaN,NaN,NaN
AT5G20960,fc00006,NaN,NaN,NaN,NaN,NaN
AT2G30720,fc00007,NaN,NaN,NaN,NaN,NaN


In [82]:
pss_fcs_merged = pss_fcs.explode("ath_homologues").set_index("ath_homologues").join(df_all, rsuffix="-dup", how="left")

pss_fcs_merged = pss_fcs_merged[sorted(pss_fcs_merged.columns)]
pss_fcs_merged.head()

,functional_cluster_id,mdo_homologues,nta_homologues,osa_homologues,parm_homologues,pavi_homologues,pcer_homologues,pcox_homologues,pdul_homologues,ppe_homologues,psib_homologues,sly_homologues,sly_homologues-dup,stu_homologues,stu_homologues-dup,vvi_homologues,vvi_homologues-dup
ath_homologues,,,,,,,,,,,,,,,,,
AT2G39350,fc00005,"[MALDO.HC.V1A1.CH1A.G25430, MALDO.HC.V1A1.CH7A...",NaN,NaN,[PRUARM.2G393500],[FUN_012043],"[PCER_051918-RA, PCER_070287-RA, PCER_075054-RA]",[PYRCO.DA.V2A1.CHR7A.173440],[TEXASF1_G9324],[PRUPE.2G224800],[PAF106G0200009667],NaN,"[SOLYC04T000402, SOLYC04T000403, SOLYC05T002605]",NaN,[SOLTU.DM.05G025440],NaN,[VITVI05_01CHR13G03040]
AT1G04580,fc00006,[],NaN,NaN,[],[],[],[],[],[],[],NaN,[SOLYC01T002518],NaN,"[SOLTU.DM.01G026450, SOLTU.DM.01G026470]",NaN,"[VITVI05_01CHR18G32770, VITVI05_01CHR18G32800]"
AT2G27150,fc00006,[MALDO.HC.V1A1.CH3A.G30687],NaN,NaN,[],[FUN_020100],"[PCER_017327-RA, PCER_020874-RA]","[PYRCO.DA.V2A1.AUGUSTUS.274070, PYRCO.DA.V2A1....",[],[],[],NaN,[SOLYC01T000347],NaN,"[SOLTU.DM.01G006270, SOLTU.DM.01G026450]",NaN,[VITVI05_01CHR06G13810]
AT5G20960,fc00006,"[MALDO.HC.V1A1.CH11A.G04924, MALDO.HC.V1A1.CH1...",NaN,NaN,[PRUARM.6G195800],[FUN_020100],"[PCER_017327-RA, PCER_020874-RA, PCER_042930-RA]","[PYRCO.DA.V2A1.AUGUSTUS.274070, PYRCO.DA.V2A1....",[TEXASF1_G21620],[PRUPE.6G150900],[PAF106G0600023214],NaN,"[SOLYC01T000347, SOLYC11T002421, SOLYC11T002422]",NaN,"[SOLTU.DM.01G006270, SOLTU.DM.11G024430, SOLTU...",NaN,[VITVI05_01CHR18G32800]
AT2G30720,fc00007,"[MALDO.HC.V1A1.CH15A.G15713, MALDO.HC.V1A1.CH2...",NaN,NaN,[],[],[],[],[TEXASF1_G27555],[PRUPE.7G267600],[],NaN,"[SOLYC12T000944, SOLYC12T002628]",NaN,"[SOLTU.DM.12G003460, SOLTU.DM.12G019560]",NaN,[VITVI05_01CHR05G25860]


In [83]:
pss_fcs_merged.columns

Index(['functional_cluster_id', 'mdo_homologues', 'nta_homologues',
       'osa_homologues', 'parm_homologues', 'pavi_homologues',
       'pcer_homologues', 'pcox_homologues', 'pdul_homologues',
       'ppe_homologues', 'psib_homologues', 'sly_homologues',
       'sly_homologues-dup', 'stu_homologues', 'stu_homologues-dup',
       'vvi_homologues', 'vvi_homologues-dup'],
      dtype='str')

In [84]:
pairs = [
    (c, c.replace("-dup", ""))
    for c in pss_fcs_merged.columns
    if c.endswith("-dup") and c.replace("-dup", "") in pss_fcs_merged.columns
]

print(pairs)

[('sly_homologues-dup', 'sly_homologues'), ('stu_homologues-dup', 'stu_homologues'), ('vvi_homologues-dup', 'vvi_homologues')]


In [85]:
def merge_dup_columns(df: pd.DataFrame) -> pd.DataFrame:
    def to_list(x):
        return x if isinstance(x, list) else []

    dup_cols = [c for c in df.columns if c.endswith("-dup")]

    for dup in dup_cols:
        base = dup[:-4]  # remove "-dup"
        if base not in df.columns:
            continue

        print(dup, base, "merging")
        df[base] = [
            list(set(to_list(a) + to_list(b)))
            for a, b in zip(df[base], df[dup])
        ]

    # drop duplicate columns
    df = df.drop(columns=dup_cols)

    return df

In [86]:
pss_fcs_merged = merge_dup_columns(pss_fcs_merged)
pss_fcs_merged.head()

sly_homologues-dup sly_homologues merging
stu_homologues-dup stu_homologues merging
vvi_homologues-dup vvi_homologues merging


,functional_cluster_id,mdo_homologues,nta_homologues,osa_homologues,parm_homologues,pavi_homologues,pcer_homologues,pcox_homologues,pdul_homologues,ppe_homologues,psib_homologues,sly_homologues,stu_homologues,vvi_homologues
ath_homologues,,,,,,,,,,,,,,
AT2G39350,fc00005,"[MALDO.HC.V1A1.CH1A.G25430, MALDO.HC.V1A1.CH7A...",NaN,NaN,[PRUARM.2G393500],[FUN_012043],"[PCER_051918-RA, PCER_070287-RA, PCER_075054-RA]",[PYRCO.DA.V2A1.CHR7A.173440],[TEXASF1_G9324],[PRUPE.2G224800],[PAF106G0200009667],"[SOLYC05T002605, SOLYC04T000403, SOLYC04T000402]",[SOLTU.DM.05G025440],[VITVI05_01CHR13G03040]
AT1G04580,fc00006,[],NaN,NaN,[],[],[],[],[],[],[],[SOLYC01T002518],"[SOLTU.DM.01G026470, SOLTU.DM.01G026450]","[VITVI05_01CHR18G32770, VITVI05_01CHR18G32800]"
AT2G27150,fc00006,[MALDO.HC.V1A1.CH3A.G30687],NaN,NaN,[],[FUN_020100],"[PCER_017327-RA, PCER_020874-RA]","[PYRCO.DA.V2A1.AUGUSTUS.274070, PYRCO.DA.V2A1....",[],[],[],[SOLYC01T000347],"[SOLTU.DM.01G006270, SOLTU.DM.01G026450]",[VITVI05_01CHR06G13810]
AT5G20960,fc00006,"[MALDO.HC.V1A1.CH11A.G04924, MALDO.HC.V1A1.CH1...",NaN,NaN,[PRUARM.6G195800],[FUN_020100],"[PCER_017327-RA, PCER_020874-RA, PCER_042930-RA]","[PYRCO.DA.V2A1.AUGUSTUS.274070, PYRCO.DA.V2A1....",[TEXASF1_G21620],[PRUPE.6G150900],[PAF106G0600023214],"[SOLYC11T002422, SOLYC11T002421, SOLYC01T000347]","[SOLTU.DM.11G024430, SOLTU.DM.11G024440, SOLTU...",[VITVI05_01CHR18G32800]
AT2G30720,fc00007,"[MALDO.HC.V1A1.CH15A.G15713, MALDO.HC.V1A1.CH2...",NaN,NaN,[],[],[],[],[TEXASF1_G27555],[PRUPE.7G267600],[],"[SOLYC12T002628, SOLYC12T000944]","[SOLTU.DM.12G003460, SOLTU.DM.12G019560]",[VITVI05_01CHR05G25860]


In [87]:
def safe_list(x):
    if isinstance(x, list):
        return sorted(x)
    if isinstance(x, str):
        return [x]
    return []

def merge_lists(series):
    out = []
    for x in series:
        if isinstance(x, list):
            out.extend(x)
    return list(set(out))


def groupby_agg_lists(df, group_col):
    df = df.copy()

    # normalize all list-like columns
    obj_cols = df.columns.difference([group_col])
    for c in obj_cols:
        df[c] = df[c].apply(safe_list)

    # group + aggregate
    return df.groupby(group_col, as_index=False).agg(merge_lists)


In [88]:
pss_new_homologues = groupby_agg_lists(pss_fcs_merged.reset_index(), "functional_cluster_id")
pss_new_homologues.head()

,functional_cluster_id,ath_homologues,mdo_homologues,nta_homologues,osa_homologues,parm_homologues,pavi_homologues,pcer_homologues,pcox_homologues,pdul_homologues,ppe_homologues,psib_homologues,sly_homologues,stu_homologues,vvi_homologues
0,fc00005,[AT2G39350],"[MALDO.HC.V1A1.CH1A.G25430, MALDO.HC.V1A1.CH7A...",[],[],[PRUARM.2G393500],[FUN_012043],"[PCER_075054-RA, PCER_070287-RA, PCER_051918-RA]",[PYRCO.DA.V2A1.CHR7A.173440],[TEXASF1_G9324],[PRUPE.2G224800],[PAF106G0200009667],"[SOLYC05T002605, SOLYC04T000403, SOLYC04T000402]",[SOLTU.DM.05G025440],[VITVI05_01CHR13G03040]
1,fc00006,"[AT5G20960, AT2G27150, AT1G04580]","[MALDO.HC.V1A1.CH3A.G30687, MALDO.HC.V1A1.CH11...",[],[],[PRUARM.6G195800],[FUN_020100],"[PCER_042930-RA, PCER_020874-RA, PCER_017327-RA]","[PYRCO.DA.V2A1.AUGUSTUS.274070, PYRCO.DA.V2A1....",[TEXASF1_G21620],[PRUPE.6G150900],[PAF106G0600023214],"[SOLYC11T002422, SOLYC11T002421, SOLYC01T00251...","[SOLTU.DM.01G026470, SOLTU.DM.11G024430, SOLTU...","[VITVI05_01CHR06G13810, VITVI05_01CHR18G32770,..."
2,fc00007,"[AT2G30720, AT5G48370]","[MALDO.HC.V1A1.CH10A.G01479, MALDO.HC.V1A1.CH2...",[],[],"[PRUARM.8G274000, PRUARM.8G273900]",[],"[PCER_090946-RA, PCER_053267-RA, PCER_053268-R...",[PYRCO.DA.V2A1.CHR5A.054460],"[TEXASF1_G29336, TEXASF1_G27555]","[PRUPE.8G185500, PRUPE.7G267600]",[],"[SOLYC09T001087, SOLYC12T002628, SOLYC12T000944]","[SOLTU.DM.12G003460, SOLTU.DM.12G019560]","[VITVI05_01CHR07G20600, VITVI05_01CHR05G25860,..."
3,fc00008,"[AT1G77330, AT1G05010, AT2G19590, AT1G12010, A...","[MALDO.HC.V1A1.CH17A.G22446, MALDO.HC.V1A1.CH5...",[],[],"[PRUARM.3G309700, PRUARM.7G327200, PRUARM.1G69...","[FUN_006690, FUN_039268, FUN_031640, FUN_01663...","[PCER_067323-RA, PCER_064731-RA, PCER_095663-R...","[PYRCO.DA.V2A1.CHR5A.070850, PYRCO.DA.V2A1.CHR...","[TEXASF1_G5791, TEXASF1_G14031, TEXASF1_G26430...","[PRUPE.7G212000, PRUPE.1G490000, PRUPE.4G01380...","[PAF106G0100005680, PAF106G0100005681, PAF106G...","[SOLYC07T001863, SOLYC07T001071, SOLYC02T00177...","[SOTUB07G018820.1.1, SOLTU.DM.07G016750, SOLTU...","[VITVI05_01CHR10G02480, VITVI05_01CHR12G07310,..."
4,fc00009,"[AT3G61510, AT4G08040, AT4G37770, AT5G65800, A...","[MALDO.HC.V1A1.CH15A.G16261, MALDO.HC.V1A1.CH1...",[],[],"[PRUARM.7G329000, PRUARM.6G328000, PRUARM.1G61...","[FUN_005791, FUN_011485, FUN_025165, FUN_03928...","[PCER_038511-RA, PCER_062604-RA, PCER_069829-R...","[PYRCO.DA.V2A1.CHR1A.343890, PYRCO.DA.V2A1.CHR...","[TEXASF1_G18551, TEXASF1_G5038, TEXASF1_G8820,...","[PRUPE.5G106200, PRUPE.2G176900, PRUPE.7G21380...","[PAF106G0600024117, PAF106G0700026585, PAF106G...","[SOLYC07T001090, SOLYC02T002720, SOLYC05T00210...","[SOLTU.DM.05G019660, SOLTU.DM.08G004500, SOLTU...","[VITVI05_01CHR07G25640, VITVI05_01CHR02G00380,..."


In [89]:
# confirm ath has not changed... 
a = pss_new_homologues["ath_homologues"].apply(safe_list)
b = pss_fcs["ath_homologues"].apply(safe_list)

mismatch = a != b

pss_diff = pd.DataFrame({
    "new": a[mismatch],
    "fcs": b[mismatch]
})

print(pss_diff)

Empty DataFrame
Columns: [new, fcs]
Index: []


In [90]:
def save_tsv_with_list_cols(df: pd.DataFrame, path: str, **kwargs):
    df_out = df.copy()

    for c in df_out.columns:
        df_out[c] = df_out[c].apply(
            lambda x: ",".join(map(str, x)) if isinstance(x, list) else x
        )

    df_out.to_csv(path, **kwargs)

In [91]:
save_tsv_with_list_cols(pss_new_homologues, skm_pss_translations_dir / f"ath_translations_per_fc_{TRANSLATION_VERSION}.tsv", sep="\t", index=False)

### Cypher code to import into PSS-neo4j


